<a href="https://colab.research.google.com/github/harshitmogre/advanced-todo-tracker/blob/main/Advanced_ANOVA_Assignment_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Statistics Assignment: Advanced ANOVA Methods
**Topics:** Repeated Measures ANOVA, Mixed ANOVA, and Welch’s ANOVA

This notebook calculates the numerical answers using Python. The PDF report contains the written answers and interpretation. Significance level: **α = 0.05**.

## Part A — Repeated Measures ANOVA
The same five employees were measured at three time points. The repeated observations are kept in the same row so the participant effect can be separated from condition effects.

In [ ]:
import numpy as np
import pandas as pd
 from scipy.stats import f

scores = np.array([[50,58,62],
                     [55,60,64],
                     [48,54,59],
                    [60,65,68],
                     [52,57,61]
                   ],dtype=float)

n, k = scores.shape
 grand_mean = scores.mean()
  condition_means = scores.mean(axis=0)
participant_means = scores.mean(axis=1)

SST = ((scores - grand_mean) ** 2).sum()
SSC = n * ((condition_means - grand_mean) ** 2).sum()
SSS = k * ((participant_means - grand_mean) ** 2).sum()
SSE = SST - SSC - SSS

df_condition = k - 1
df_error = (n - 1) * (k - 1)
MS_condition = SSC / df_condition
MS_error = SSE / df_error
F_stat = MS_condition / MS_error
p_value = f.sf(F_stat, df_condition, df_error)

print("Condition means:", condition_means)
print("Grand mean:", grand_mean)
print(f"SST={SST:.4f}, SSC={SSC:.4f}, SSS={SSS:.4f}, SSE={SSE:.4f}")
print(f"df condition={df_condition}, df error={df_error}")
print(f"MS condition={MS_condition:.4f}, MS error={MS_error:.4f}")
print(f"F({df_condition}, {df_error})={F_stat:.4f}, p={p_value:.8g}")

## Part B — Mixed ANOVA
This is a 2 × 2 mixed design: Group (Treatment/Control) is between subjects, and Time (Before/After) is within subjects. The code calculates the classical balanced mixed-ANOVA sums of squares and tests the group, time, and interaction effects.

In [ ]:
from scipy.stats import f, levene, shapiro

treatment=np.array([[50,60],[52,63],[48,58],[55,65],[45,55]],dtype=float)
control=np.array([[50,52],[53,54],[47,49],[52,53],[48,50]],dtype=float)

records=[]
for group, arr in [("Treatment", treatment), ("Control", control)]:
    for i, (before, after) in enumerate(arr, start=1):
        records += [ {"subject": f"{group[0]}{i}", "group": group, "time": "Before", "score": before}, {"subject": f"{group[0]}{i}",
                      "group": group, "time": "After", "score": after}]
data = pd.DataFrame(records)

grand = data["score"].mean()
group_means = data.groupby("group")["score"].mean()
time_means = data.groupby("time")["score"].mean()
cell_means = data.groupby(["group", "time"])["score"].mean()

subject_means = data.groupby("subject")["score"].mean()

a, b, n_per_group = 2, 2, 5
SS_group = b * n_per_group * sum((group_means[g] - grand)**2 for g in group_means.index)
SS_subject_group = 0.0
for group, arr in [("Treatment", treatment), ("Control", control)]:
    for values in arr:
        SS_subject_group += b * (values.mean()-group_means[group])**2

SS_time = a * n_per_group * sum((time_means[t]-grand)**2 for t in time_means.index)
SS_interaction = 0.0
for g in group_means.index:
    for t in time_means.index:
        SS_interaction+=n_per_group * (
            cell_means[g, t]-group_means[g]-time_means[t]+grand)**2

SS_total = ((data["score"]-grand)**2).sum()
SS_within_error = SS_total - SS_group - SS_subject_group - SS_time - SS_interaction

df_group, df_subject_group = 1, 8
df_time, df_interaction, df_within_error = 1, 1, 8
F_group = (SS_group / df_group) / (SS_subject_group / df_subject_group)
F_time = (SS_time / df_time) / (SS_within_error / df_within_error)
F_interaction = (SS_interaction / df_interaction) / (SS_within_error / df_within_error)

p_group = f.sf(F_group, df_group, df_subject_group)
p_time = f.sf(F_time, df_time, df_within_error)
p_interaction = f.sf(F_interaction, df_interaction, df_within_error)

print("Group means (averaged over time):\n", group_means)
print("Time means (averaged over groups):\n", time_means)
print("Group-by-time cell means:\n", cell_means)
print(f"Group: F({df_group}, {df_subject_group})={F_group:.4f}, p={p_group:.6f}")
print(f"Time: F({df_time}, {df_within_error})={F_time:.4f}, p={p_time:.8g}")
print(f"Group × Time: F({df_interaction}, {df_within_error})={F_interaction:.4f}, p={p_interaction:.8g}")

# Basic diagnostics; with only five participants per group, interpret cautiously.
print("Levene test across groups:", levene(treatment.ravel(), control.ravel()))
print("Shapiro-Wilk test of full-model cell residuals:")
data["fitted"] = data.apply(lambda r: cell_means[r["group"], r["time"]], axis=1)
print(shapiro(data["score"] - data["fitted"]))
print("Sphericity is automatically satisfied because Time has only two levels.")

## Part C — Welch’s ANOVA
Welch’s ANOVA is appropriate for independent groups when equal variances cannot reasonably be assumed. Sample variance below uses the usual sample denominator (n − 1).

In [ ]:
from scipy.stats import f

class_a = np.array([45,50,55,60], dtype=float)
class_b = np.array([55,60,65,70,75], dtype=float)
class_c = np.array([40,60,80,100,120,140], dtype=float)
groups = [class_a,class_b,class_c]

ns = np.array([len(g) for g in groups], dtype=float)
means = np.array([g.mean() for g in groups])
variances = np.array([g.var(ddof=1) for g in groups])
weights = ns / variances
W = weights.sum()
weighted_mean = np.sum(weights * means) / W
A = sum((1 / (ns[i] - 1)) * (1 - weights[i] / W)**2 for i in range(len(groups)))

k = len(groups)
numerator = np.sum(weights * (means - weighted_mean)**2) / (k - 1)
denominator = 1 + (2 * (k - 2) / (k**2 - 1)) * A
F_welch = numerator / denominator
df1 = k - 1
df2 = (k**2 - 1) / (3 * A)
p_welch = f.sf(F_welch, df1, df2)

summary = pd.DataFrame({
    "Class": ["A", "B", "C"],
    "n": ns.astype(int),
    "Mean": means,
    "Sample variance": variances,
    "Weight": weights
})
print(summary.to_string(index=False))
print(f"Total weight W={W:.6f}")
print(f"Weighted grand mean={weighted_mean:.6f}")
print(f"Correction A={A:.8f}")
print(f"Welch F({df1}, {df2:.4f})={F_welch:.4f}, p={p_welch:.6f}")
print("Decision at alpha=0.05:", "Reject H0" if p_welch < 0.05 else "Fail to reject H0")

## Part D — Comparative Understanding
1. Repeated Measures ANOVA.
2. Mixed ANOVA, particularly its Group × Time interaction.
3. Welch’s ANOVA.
4. Ordinary One-Way ANOVA assumes equal population variances; Welch’s ANOVA adjusts the statistic and degrees of freedom for unequal variances.
5. Repeated observations come from the same participant and are generally correlated; independent observations come from different participants and are assumed unrelated across observations.